<a href="https://colab.research.google.com/github/limagabriela/PyroCb/blob/main/fire_evolution_pyrocb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PyroCb — Fire evolution (NASA FIRMS)

Fonte de edição do módulo `Fire_evolution_pyrocb.py`. Cada bloco com `# EXPORT_TO_GITHUB` será copiado, na ordem, para o módulo Python pelo GitHub Actions.

**Organização:** imports → parâmetros das figuras → API → download/filtros → série temporal → mapa → `baixar_firms()`.

A chave `MAP_KEY` **não** fica armazenada neste notebook; ela é fornecida exclusivamente pelo notebook chamador. Os gráficos são exportados em PDF vetorial e PNG em `/content/firms/figures`. Por padrão, **não** substituem arquivos existentes; para reexportar a mesma consulta após alterar o estilo, defina `OVERWRITE_FIGURES = True` no bloco dos parâmetros.

A identificação das figuras inclui região, período solicitado e confiança MODIS. O download consulta MODIS e VIIRS e filtra apenas depois de reunir os resultados.

## 1. Imports e fontes FIRMS

In [ ]:
# EXPORT_TO_GITHUB
"""Funções do projeto PyroCb para baixar, filtrar e plotar dados do FIRMS.

Este código é exportado automaticamente para Fire_evolution_pyrocb.py.
Não armazene a MAP_KEY aqui.
"""

from datetime import date, timedelta
from io import StringIO
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
import requests

FAMILIAS = ("MODIS", "VIIRS_SNPP", "VIIRS_NOAA20", "VIIRS_NOAA21")
BASE_URL = "https://firms.modaps.eosdis.nasa.gov/api"

## 2. Parâmetros gerais das figuras

In [ ]:
# EXPORT_TO_GITHUB
# ==================================================
# PUBLICATION FIGURE SETTINGS — NATURE-LIKE
# ==================================================

import math
import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from pathlib import Path

# --------------------------------------------------
# FIGURE DIMENSIONS
# Nature: 89 mm (single column), 183 mm (double)
# --------------------------------------------------

MM = 1 / 25.4

FIG_WIDTH_MM = 183
FIG_SERIES_HEIGHT_MM = 75
FIG_MAP_HEIGHT_MM = 125

FIG_SERIES_SIZE = (
    FIG_WIDTH_MM * MM,
    FIG_SERIES_HEIGHT_MM * MM
)

FIG_MAP_SIZE = (
    FIG_WIDTH_MM * MM,
    FIG_MAP_HEIGHT_MM * MM
)

# --------------------------------------------------
# EXPORT
# --------------------------------------------------

FIG_DIR = Path("/content/firms/figures")

SAVE_FIGURES = True
OVERWRITE_FIGURES = False

FIG_DPI = 450

# --------------------------------------------------
# ACCESSIBLE COLOR PALETTE — WONG
# --------------------------------------------------

COLORS = {
    "blue": "#0072B2",
    "orange": "#E69F00",
    "vermillion": "#D55E00",
    "green": "#009E73",
    "purple": "#CC79A7",
    "black": "#222222",
    "gray": "#737373",
    "light_gray": "#D9D9D9",
    "white": "#FFFFFF",
}

COLOR_SERIES = COLORS["blue"]
COLOR_MODIS = COLORS["vermillion"]
COLOR_VIIRS = COLORS["blue"]

# --------------------------------------------------
# MATPLOTLIB STYLE
# --------------------------------------------------

mpl.rcParams.update({

    # Fonts
    "font.family": "sans-serif",
    "font.sans-serif": [
        "Arial",
        "Helvetica",
        "Liberation Sans",
        "DejaVu Sans"
    ],

    "font.size": 7,
    "axes.labelsize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "legend.fontsize": 6.5,

    # Axes
    "axes.linewidth": 0.75,
    "axes.edgecolor": COLORS["black"],
    "axes.labelcolor": COLORS["black"],

    "axes.spines.top": False,
    "axes.spines.right": False,

    # Ticks
    "xtick.direction": "out",
    "ytick.direction": "out",

    "xtick.major.size": 3,
    "ytick.major.size": 3,

    "xtick.major.width": 0.65,
    "ytick.major.width": 0.65,

    "xtick.color": COLORS["black"],
    "ytick.color": COLORS["black"],

    # Lines
    "lines.linewidth": 1.0,
    "lines.markersize": 3,

    # Background
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",

    # No decorative grid
    "axes.grid": False,

    # Legend
    "legend.frameon": False,

    # PDF / SVG
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",

    # Display
    "figure.dpi": 140,
    "savefig.dpi": FIG_DPI,
})


# --------------------------------------------------
# FIGURE EXPORT FUNCTION
# --------------------------------------------------

def save_publication_figure(fig, name):

    if not SAVE_FIGURES:
        return

    FIG_DIR.mkdir(parents=True, exist_ok=True)

    for ext in ("pdf", "png"):

        path = FIG_DIR / f"{name}.{ext}"

        if path.exists() and not OVERWRITE_FIGURES:
            print(f"Figure already exists: {path}")
            continue

        fig.savefig(
            path,
            dpi=FIG_DPI,
            facecolor="white"
        )

        print(f"Figure saved: {path}")

# --------------------------------------------------
# SAFE OUTPUT NAME IDENTIFIER
# --------------------------------------------------

def _id_area(bbox):
    """Convert a bounding box to a stable file-name identifier."""
    return (str(bbox).lower().strip().replace(" ", "")
            .replace(",", "_").replace("-", "m").replace(".", "p"))

## 3. Funções auxiliares da API

In [ ]:
# EXPORT_TO_GITHUB
# ==================================================
# FUNÇÕES AUXILIARES DA API
# ==================================================

def _fonte_disponivel(linhas, familia, dia):
    """Prefere SP e usa NRT apenas nas datas ainda sem SP."""
    for modo in ("SP", "NRT"):
        nome = f"{familia}_{modo}"
        if any(
            item["data_id"] == nome and item["min_date"] <= dia <= item["max_date"]
            for item in linhas
        ):
            return nome
    return None


def _montar_tarefas(linhas, inicio, fim):
    """Monta requisições de até 5 dias para cada família instrumental."""
    tarefas = []
    cobertura = set()
    for familia in FAMILIAS:
        dia = inicio
        while dia <= fim:
            fonte = _fonte_disponivel(linhas, familia, dia)
            if fonte is None:
                dia += timedelta(days=1)
                continue
            primeiro = dia
            dias = 0
            while dia <= fim and dias < 5 and _fonte_disponivel(linhas, familia, dia) == fonte:
                cobertura.add(dia)
                dia += timedelta(days=1)
                dias += 1
            tarefas.append((fonte, primeiro, dias))
    return tarefas, cobertura


def _ler_csv_firms(resposta, fonte):
    """Interpreta a resposta da API FIRMS."""
    if resposta.status_code != 200:
        raise RuntimeError(f"Erro HTTP {resposta.status_code} consultando {fonte}.")

    conteudo = resposta.text.strip()
    if conteudo.lower().startswith(("no data", "no records")):
        return pd.DataFrame(columns=["latitude", "longitude", "confidence"])
    if not conteudo:
        raise RuntimeError(f"Resposta vazia da API para {fonte}.")

    df = pd.read_csv(StringIO(conteudo), dtype={"confidence": "string"})
    colunas_minimas = {"latitude", "longitude", "confidence"}
    if not colunas_minimas.issubset(df.columns):
        raise RuntimeError(f"Resposta inesperada da API ({fonte}): {conteudo[:200]}")
    return df


def _garantir_pyarrow():
    try:
        import pyarrow  # noqa: F401
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyarrow"])

## 4. Download, filtro e Parquet

In [ ]:
# EXPORT_TO_GITHUB
# ==================================================
# DOWNLOAD E PROCESSAMENTO DOS DADOS
# ==================================================

def baixar_e_processar_firms(
    map_key,
    data_inicial="2019-08-16",
    data_final="2019-08-19",
    bbox="-59,-21,-53,-17",
    confianca_min=50,
    pasta="/content/firms/dados",
    rebaixar=False,
):
    """Baixa MODIS + VIIRS, filtra por confiança e salva um Parquet.

    Returns:
        dict com df_firms, arquivo, resumo e datas.
    """
    if not str(map_key).strip():
        raise ValueError("Forneça map_key no notebook que chama baixar_e_processar_firms().")

    _garantir_pyarrow()

    inicio = date.fromisoformat(data_inicial)
    fim = date.fromisoformat(data_final)
    if inicio > fim:
        raise ValueError("data_inicial deve ser anterior ou igual a data_final")

    pasta = Path(pasta)
    pasta.mkdir(parents=True, exist_ok=True)
    area_id = _id_area(bbox)
    arquivo = pasta / (
        f"FIRMS_MODIS_VIIRS_{data_inicial}_{data_final}_{area_id}_"
        f"modis{confianca_min}_viirsNH.parquet"
    )

    if arquivo.exists() and not rebaixar:
        try:
            df_firms = pd.read_parquet(arquivo)
            if df_firms.empty or not {"sensor", "fonte_firms"}.issubset(df_firms.columns):
                raise ValueError("Parquet vazio ou sem colunas necessárias")
            resumo = pd.DataFrame()
            print(f"Reutilizando: {arquivo} ({len(df_firms):,} focos)")
            return {
                "df_firms": df_firms,
                "arquivo": arquivo,
                "resumo": resumo,
                "inicio": inicio,
                "fim": fim,
                "bbox": bbox,
            }
        except Exception as exc:
            raise RuntimeError(
                f"Parquet existente inválido: {arquivo}. Use rebaixar=True para refazer."
            ) from exc

    registros = []
    with requests.Session() as sess:
        disponibilidade = sess.get(f"{BASE_URL}/data_availability/csv/{map_key}/ALL", timeout=120)
        disponibilidade.raise_for_status()
        tab = pd.read_csv(StringIO(disponibilidade.text))

        colunas = {"data_id", "min_date", "max_date"}
        if not colunas.issubset(tab.columns):
            raise RuntimeError(f"Resposta inesperada da disponibilidade: {disponibilidade.text[:200]}")

        tab["data_id"] = tab["data_id"].astype(str).str.strip().str.upper()
        tab["min_date"] = pd.to_datetime(tab["min_date"]).dt.date
        tab["max_date"] = pd.to_datetime(tab["max_date"]).dt.date

        linhas = tab.to_dict("records")
        tarefas, cobertura = _montar_tarefas(linhas, inicio, fim)

        todos_dias = {inicio + timedelta(days=i) for i in range((fim - inicio).days + 1)}
        faltantes = sorted(todos_dias - cobertura)
        if faltantes:
            raise RuntimeError(
                f"Sem cobertura MODIS/VIIRS para {len(faltantes)} dia(s), por exemplo {faltantes[:3]}."
            )

        print("Fontes consultadas:", ", ".join(dict.fromkeys(t[0] for t in tarefas)))
        print("A filtragem de confiança ocorre somente depois de juntar todas as fontes.\n")

        for fonte, primeiro, dias in tarefas:
            url = f"{BASE_URL}/area/csv/{map_key}/{fonte}/{bbox}/{dias}/{primeiro.isoformat()}"
            df = _ler_csv_firms(sess.get(url, timeout=240), fonte)
            df["fonte_firms"] = fonte
            df["sensor"] = "MODIS" if fonte.startswith("MODIS") else "VIIRS"
            registros.append(df)
            print(f"{fonte:20s} {primeiro.isoformat()} (+{dias-1}): {len(df):,} recebidos")

    df_bruto = pd.concat(registros, ignore_index=True)
    conf = df_bruto["confidence"].astype("string").str.strip().str.lower()
    conf_num = pd.to_numeric(conf.str.replace("%", "", regex=False), errors="coerce")

    filtro_modis = df_bruto["sensor"].eq("MODIS") & conf_num.ge(confianca_min)
    filtro_viirs = df_bruto["sensor"].eq("VIIRS") & conf.isin(["n", "nominal", "h", "high"])
    df_firms = df_bruto.loc[(filtro_modis | filtro_viirs).fillna(False)].copy().reset_index(drop=True)

    resumo = (
        df_bruto.groupby("fonte_firms").size().rename("recebidos").to_frame()
        .join(df_firms.groupby("fonte_firms").size().rename("mantidos"))
        .fillna(0).astype(int)
    )

    print("\n=========== RESULTADO DO FILTRO ===========")
    print(resumo.to_string())
    print(f"Total bruto: {len(df_bruto):,} | Total filtrado: {len(df_firms):,}")

    if df_firms.empty:
        raise RuntimeError("Nenhum foco passou no filtro; nenhum Parquet foi salvo.")

    df_firms.to_parquet(arquivo, engine="pyarrow", compression="zstd", index=False)
    print(f"\nParquet salvo em: {arquivo}")

    return {
        "df_firms": df_firms,
        "arquivo": arquivo,
        "resumo": resumo,
        "inicio": inicio,
        "fim": fim,
        "bbox": bbox,
    }

## 5. Figura 1 — Série temporal

In [ ]:

# EXPORT_TO_GITHUB
# ==================================================
# FIGURE 1 — HOURLY FIRE EVOLUTION (FOUR PANELS)
# ==================================================

def gerar_figura_serie_temporal(
    df_firms,
    inicio=None,
    fim=None,
    mostrar=True,
    bbox="world",
    confianca_min=50,
):
    """Hourly FIRMS counts, FRP and sensor-specific brightness temperatures."""

    # Acquisition datetime: 233 means 02:33 UTC (not 23:30).
    df = df_firms.copy()
    dates = pd.to_datetime(df["acq_date"], errors="coerce")
    hhmm = (
        pd.to_numeric(df["acq_time"], errors="coerce")
        .astype("Int64").astype("string").str.zfill(4)
    )
    df["datetime_utc"] = pd.to_datetime(
        dates.dt.strftime("%Y-%m-%d") + " "
        + hhmm.str[:2] + ":" + hhmm.str[2:],
        format="%Y-%m-%d %H:%M",
        errors="coerce",
    )
    df = df.dropna(subset=["datetime_utc"])
    if df.empty:
        raise ValueError("No valid FIRMS acquisition timestamps.")

    # Data quality: FRP in MW; brightness temperature in kelvin.
    df["frp_mw"] = pd.to_numeric(df["frp"], errors="coerce")
    df.loc[df["frp_mw"] < 0, "frp_mw"] = float("nan")
    df["temp_k"] = float("nan")
    for sensor, column in (("MODIS", "brightness"), ("VIIRS", "bright_ti4")):
        if column in df.columns:
            mask = df["sensor"].eq(sensor)
            df.loc[mask, "temp_k"] = pd.to_numeric(
                df.loc[mask, column], errors="coerce"
            )
    df.loc[df["temp_k"] <= 0, "temp_k"] = float("nan")
    df["temp_c"] = df["temp_k"] - 273.15

    # Bins with no detections have zero counts, but undefined FRP/temperature.
    df["bin_utc"] = df["datetime_utc"].dt.floor(f"{TIME_BIN_HOURS}h")
    inicio_data = pd.Timestamp(inicio).normalize() if inicio is not None else df["bin_utc"].min().normalize()
    fim_data = pd.Timestamp(fim).normalize() if fim is not None else df["bin_utc"].max().normalize()
    if inicio_data > fim_data:
        raise ValueError("Start date is after end date.")
    horario = pd.date_range(
        inicio_data,
        fim_data + pd.Timedelta(days=1) - pd.Timedelta(hours=TIME_BIN_HOURS),
        freq=f"{TIME_BIN_HOURS}h",
    )

    serie_temporal = (
        df.groupby("bin_utc")
        .agg(
            focos=("sensor", "size"),
            frp_max_mw=("frp_mw", "max"),
            frp_mean_mw=("frp_mw", "mean"),
        )
        .reindex(horario)
        .rename_axis("data")
    )
    serie_temporal["focos"] = serie_temporal["focos"].fillna(0).astype(int)

    # Do not pool MODIS and VIIRS temperatures (different spectral channels).
    temp = (
        df.groupby(["bin_utc", "sensor"])["temp_c"]
        .agg(["mean", "max"])
        .unstack("sensor")
        .reindex(horario)
    )
    for sensor in ("MODIS", "VIIRS"):
        for stat in ("mean", "max"):
            key = (stat, sensor)
            serie_temporal[f"temp_{sensor.lower()}_{stat}_c"] = (
                temp[key] if key in temp.columns else float("nan")
            )
    serie_temporal = serie_temporal.reset_index()

    # Nature-like multi-panel figure: 183 mm wide for two-column LaTeX.
    fig, axes = plt.subplots(
        4, 1,
        figsize=FIG_SERIES_SIZE,
        sharex=True,
        gridspec_kw={"height_ratios": [1, 1, 1, 1.25]},
        layout="constrained",
    )
    x = serie_temporal["data"]
    bin_width = TIME_BIN_HOURS / 24.0
    kw = dict(linewidth=0, align="center", rasterized=False)

    axes[0].bar(x, serie_temporal["focos"], width=bin_width * 0.83,
                color=COLORS["blue"], **kw)
    axes[1].bar(x, serie_temporal["frp_max_mw"], width=bin_width * 0.83,
                color=COLORS["vermillion"], **kw)
    axes[2].bar(x, serie_temporal["frp_mean_mw"], width=bin_width * 0.83,
                color=COLORS["green"], **kw)

    # Grouped bars: mean and maximum for each sensor, in the SAME temperature panel.
    config = (
        ("MODIS", "mean", -1.5, COLORS["orange"], 0.52),
        ("MODIS", "max", -0.5, COLORS["orange"], 1.00),
        ("VIIRS", "mean", 0.5, COLORS["blue"], 0.52),
        ("VIIRS", "max", 1.5, COLORS["blue"], 1.00),
    )
    width = bin_width * 0.21
    for sensor, stat, offset, color, alpha in config:
        axes[3].bar(
            x + pd.to_timedelta(offset * width, unit="D"),
            serie_temporal[f"temp_{sensor.lower()}_{stat}_c"],
            width=width,
            color=color,
            alpha=alpha,
            label=f"{sensor} {stat}",
            **kw,
        )

    labels = (
        "Fire detections",
        "Maximum FRP (MW)",
        "Mean FRP (MW)",
        "Brightness temperature (°C)",
    )
    for ax, label, letter in zip(axes, labels, "abcd"):
        ax.set_ylabel(label)
        ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=4))
        ax.tick_params(axis="both", which="minor", bottom=False, left=False)
        ax.text(-0.08, 1.02, letter, transform=ax.transAxes,
                fontsize=8, fontweight="bold", va="bottom")
        if ax is not axes[3]:
            ax.set_ylim(bottom=0)

    axes[0].yaxis.set_major_locator(mticker.MaxNLocator(nbins=4, integer=True))
    axes[3].legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.22),
                   columnspacing=1.0, handlelength=1.0, fontsize=6)

    # Explicit date + hour UTC on the last x-axis, no more than ~6 ticks.
    locator = mdates.AutoDateLocator(minticks=3, maxticks=6)
    axes[-1].xaxis.set_major_locator(locator)
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b\n%H:%M"))
    axes[-1].set_xlabel("Acquisition date and time (UTC)")
    axes[-1].set_xlim(
        horario[0] - pd.Timedelta(hours=TIME_BIN_HOURS / 2),
        horario[-1] + pd.Timedelta(hours=TIME_BIN_HOURS / 2),
    )

    name = (
        f"firms_fire_evolution_{inicio_data:%Y%m%d}_{fim_data:%Y%m%d}_"
        f"{_id_area(bbox)}_modis{confianca_min}_viirsNH_{TIME_BIN_HOURS}h"
    )
    save_publication_figure(fig, name)

    if mostrar:
        from IPython.display import display
        display(fig)
        plt.close(fig)

    return serie_temporal, fig


## 6. Figura 2 — Mapa de focos

In [ ]:
# EXPORT_TO_GITHUB
# ==================================================
# FIGURE 2 — SPATIAL DISTRIBUTION
# ==================================================

def gerar_figura_mapa(
    df_firms,
    bbox="world",
    mostrar=True,
    inicio=None,
    fim=None,
    confianca_min=50,
):

    # ----------------------------------------------
    # DATA
    # ----------------------------------------------

    df = df_firms.copy()

    df["latitude"] = pd.to_numeric(
        df["latitude"], errors="coerce"
    )

    df["longitude"] = pd.to_numeric(
        df["longitude"], errors="coerce"
    )

    df = df.dropna(
        subset=["latitude", "longitude"]
    )

    if df.empty:
        raise ValueError("No valid coordinates.")

    viirs = df[df["sensor"] == "VIIRS"]
    modis = df[df["sensor"] == "MODIS"]

    # ----------------------------------------------
    # FIGURE
    # ----------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIG_MAP_SIZE,
        layout="constrained"
    )

    # VIIRS — filled points
    if not viirs.empty:

        ax.scatter(
            viirs["longitude"],
            viirs["latitude"],
            s=7,
            color=COLOR_VIIRS,
            alpha=0.55,
            linewidths=0,
            label=f"VIIRS (n={len(viirs):,})",
            rasterized=len(viirs) > 3000,
            zorder=2
        )

    # MODIS — open circles
    if not modis.empty:

        ax.scatter(
            modis["longitude"],
            modis["latitude"],
            s=20,
            marker="o",
            facecolors="none",
            edgecolors=COLOR_MODIS,
            linewidths=0.7,
            alpha=0.85,
            label=f"MODIS (n={len(modis):,})",
            rasterized=len(modis) > 3000,
            zorder=3
        )

    # ----------------------------------------------
    # SPATIAL EXTENT
    # ----------------------------------------------

    if str(bbox).lower() != "world":

        west, south, east, north = map(
            float, bbox.split(",")
        )

        ax.set_xlim(west, east)
        ax.set_ylim(south, north)

        mean_lat = (south + north) / 2

    else:

        ax.set_xlim(-180, 180)
        ax.set_ylim(-90, 90)

        mean_lat = 0

    # Approximate geographic aspect ratio
    ax.set_aspect(
        1 / max(math.cos(math.radians(mean_lat)), 0.1),
        adjustable="box"
    )

    # ----------------------------------------------
    # AXES
    # ----------------------------------------------

    ax.set_xlabel("Longitude (°)")
    ax.set_ylabel("Latitude (°)")

    ax.xaxis.set_major_locator(
        mticker.MaxNLocator(nbins=5)
    )

    ax.yaxis.set_major_locator(
        mticker.MaxNLocator(nbins=5)
    )

    ax.xaxis.set_major_formatter(
        mticker.FormatStrFormatter("%.1f")
    )

    ax.yaxis.set_major_formatter(
        mticker.FormatStrFormatter("%.1f")
    )

    # Map frame
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.7)

    # ----------------------------------------------
    # LEGEND
    # ----------------------------------------------

    if ax.get_legend_handles_labels()[1]:

        ax.legend(
            loc="lower center",
            bbox_to_anchor=(0.5, 1.01),
            ncol=2,
            frameon=False,
            handletextpad=0.4,
            columnspacing=1.5
        )

    # ----------------------------------------------
    # EXPORT
    # ----------------------------------------------

    dates = pd.to_datetime(
        df["acq_date"],
        errors="coerce"
    )

    start = pd.Timestamp(inicio) if inicio is not None else dates.min()
    end = pd.Timestamp(fim) if fim is not None else dates.max()

    area_id = _id_area(bbox)

    name = (
        f"firms_spatial_distribution_"
        f"{start:%Y%m%d}_{end:%Y%m%d}_{area_id}_"
        f"modis{confianca_min}_viirsNH"
    )

    save_publication_figure(fig, name)

    if mostrar:
        from IPython.display import display
        display(fig)
        plt.close(fig)

    return fig

## 7. Função pública — chamada pelo outro Colab

In [ ]:
# EXPORT_TO_GITHUB
# ==================================================
# MAIN PUBLIC FUNCTION
# ==================================================

def baixar_firms(
    map_key,
    data_inicial="2019-08-16",
    data_final="2019-08-19",
    bbox="-59,-21,-53,-17",
    confianca_min=50,
    pasta="/content/firms/dados",
    rebaixar=False,
    mostrar=True,
):
    """Fluxo completo: download, filtro, Parquet e figuras.

    Returns:
        dict com df_firms, fig_serie, fig_mapa, serie_temporal, arquivo e resumo.
    """
    resultado = baixar_e_processar_firms(
        map_key=map_key,
        data_inicial=data_inicial,
        data_final=data_final,
        bbox=bbox,
        confianca_min=confianca_min,
        pasta=pasta,
        rebaixar=rebaixar,
    )

    serie_temporal, fig_serie = gerar_figura_serie_temporal(
        resultado["df_firms"],
        inicio=resultado["inicio"],
        fim=resultado["fim"],
        mostrar=mostrar,
        bbox=resultado["bbox"],
        confianca_min=confianca_min,
    )
    fig_mapa = gerar_figura_mapa(
        resultado["df_firms"],
        bbox=resultado["bbox"],
        mostrar=mostrar,
        inicio=resultado["inicio"],
        fim=resultado["fim"],
        confianca_min=confianca_min,
    )

    return {
        "df_firms": resultado["df_firms"],
        "fig_serie": fig_serie,
        "fig_mapa": fig_mapa,
        "serie_temporal": serie_temporal,
        "arquivo": resultado["arquivo"],
        "resumo": resultado["resumo"],
    }

## Usar em outro Colab

Salve este notebook no GitHub com o nome exato **`Fire_evolution_pyrocb.ipynb`**, na branch `main` do repositório `limagabriela/PyroCb`. O workflow `.github/workflows/export-firms.yml` gerará `Fire_evolution_pyrocb.py` automaticamente. Consulte a aba **Actions** para verificar a conclusão do commit.

No notebook que executa a análise, importe `baixar_firms` do arquivo `.py` publicado e passe a chave `map_key` diretamente na chamada. Esse notebook-fonte não executa downloads por conta própria ao selecionar *Run all*.